# M3 · Ejecución integral del sistema RAG clínico

Une el trabajo de **todo el equipo** para el Módulo 3 y corre el pipeline de
punta a punta:

```
corpus (Isa)  →  retrieval + orquestación (Pau)  →  generación (Agustín)  →  RAGAS + cruce con M2 (Luis)
```

**Funciona igual en Google Colab y en local.** El notebook detecta el entorno y
se adapta: en Colab clona la rama `dev` y monta Drive; en local usa el repositorio
donde ya está y un directorio de datos propio.

Las celdas son Python puro (sin `%pip`, `!` ni `%cd`), así que corren en cualquier
kernel (Colab, Jupyter, VS Code) y también se pueden ejecutar como script.

**Requisitos:** `GROQ_API_KEY` (generación y RAGAS) y, opcionalmente,
`BIOPORTAL_API_KEY` (tool de normalización real). En Colab van en Secretos; en
local en el entorno o en un `.env`.

## 1 · Entorno y parámetros

Detecta Colab vs. local y define las rutas. En local se puede sobrescribir con
`M3_REPO_DIR` (raíz del repo) y `M3_DATA_ROOT` (datos), si no se toman los valores
por defecto.

In [28]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


def _raiz_repo(inicio):
    p = Path(inicio).resolve()
    for cand in [p, *p.parents]:
        if (cand / "M3").is_dir() and (cand / ".git").exists():
            return cand
    return p


RAMA = "dev"
REPO_URL = "https://github.com/luisNP21/Topicos-IA.git"

if EN_COLAB:
    REPO_DIR = Path("/content/Topicos-IA")
    DATA_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3")
else:
    REPO_DIR = _raiz_repo(os.environ.get("M3_REPO_DIR") or os.getcwd())
    DATA_ROOT = Path(os.environ.get("M3_DATA_ROOT") or (REPO_DIR.parent / "m3_data")).resolve()

# Eval set del equipo (15 casos: entidad + pregunta + esperado).
EJEMPLOS = "M3/ejecucion/eval_set_casos.json"
EJEMPLOS_ALT = "M3/generacion/entidades_ejemplo.json"

MODO_RAGAS = "real"      # "real" (librería ragas + juez Groq) | "mock"
CORRER_CORPUS = True     # False si el índice ya está construido
CORRER_EXPERIMENTO = True  # calibra umbrales y mide el delta del retrieval

print("Entorno:", "Colab" if EN_COLAB else "local")
print("REPO_DIR:", REPO_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("RAGAS:", MODO_RAGAS, "| corpus:", CORRER_CORPUS, "| experimento:", CORRER_EXPERIMENTO)

Entorno: Colab
REPO_DIR: /content/Topicos-IA
DATA_ROOT: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3
RAGAS: real | corpus: True | experimento: True


## 2 · Sincronizar el repositorio

En **Colab** se clona/sincroniza `dev` (el runtime se pierde al cerrar la sesión).
En **local** se usa el repositorio tal cual, **sin** clonar ni resetear (no se
pisan cambios locales).

In [29]:
def ejecutar(cmd, cwd=None, extra_env=None, check=True):
    """Corre un comando mostrando su salida; aborta si falla."""
    env = {**os.environ, **(extra_env or {})}
    print("$", " ".join(str(c) for c in cmd))
    r = subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None,
                       env=env, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    if check and r.returncode:
        raise RuntimeError("Fallo (%d): %s" % (r.returncode, " ".join(str(c) for c in cmd)))
    return r


if EN_COLAB:
    if (REPO_DIR / ".git").exists():
        ejecutar(["git", "-C", REPO_DIR, "fetch", "origin", RAMA])
        ejecutar(["git", "-C", REPO_DIR, "checkout", "-B", RAMA, "origin/" + RAMA])
        ejecutar(["git", "-C", REPO_DIR, "reset", "--hard", "origin/" + RAMA])
    else:
        ejecutar(["git", "clone", "--branch", RAMA, REPO_URL, REPO_DIR])
    print(ejecutar(["git", "-C", REPO_DIR, "log", "-1", "--oneline"]).stdout.strip())
else:
    print("Modo local: se usa el repo en", REPO_DIR, "(no se clona ni se resetea).")

$ git -C /content/Topicos-IA fetch origin dev
From https://github.com/luisNP21/Topicos-IA
 * branch            dev        -> FETCH_HEAD
   e2688c0..9154a79  dev        -> origin/dev

$ git -C /content/Topicos-IA checkout -B dev origin/dev
branch 'dev' set up to track 'origin/dev'.
Your branch is up to date with 'origin/dev'.

Reset branch 'dev'

$ git -C /content/Topicos-IA reset --hard origin/dev
HEAD is now at 9154a79 Merge pull request #30 from luisNP21/appM4

$ git -C /content/Topicos-IA log -1 --oneline
9154a79 Merge pull request #30 from luisNP21/appM4

9154a79 Merge pull request #30 from luisNP21/appM4


## 3 · Instalar dependencias

Cada pieza trae su `requirements.txt`; el retrieval no, así que sus dependencias
se instalan aparte.

In [30]:
for req in ["M3/corpus", "M3/tools", "M3/generacion", "M3/ragas"]:
    ejecutar([sys.executable, "-m", "pip", "install", "-q", "-r", REPO_DIR / req / "requirements.txt"])

ejecutar([sys.executable, "-m", "pip", "install", "-q",
          "sentence-transformers", "chromadb", "rank_bm25", "groq", "python-dotenv", "PyYAML"])
print("Dependencias instaladas.")

$ /usr/bin/python3 -m pip install -q -r /content/Topicos-IA/M3/corpus/requirements.txt
$ /usr/bin/python3 -m pip install -q -r /content/Topicos-IA/M3/tools/requirements.txt
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.31 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.3 which is incompatible.
docling-core 2.104.0 requires requests<3.0.0,>=2.34.2, but you have requests 2.32.3 which is incompatible.
kubernetes 37.0.1 requires PyYAML>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.3 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
g

## 4 · Datos y credenciales

- **Colab:** monta Drive y toma las claves de los Secretos.
- **Local:** toma las claves del entorno o de un `.env`.

En ambos casos se escribe un `.env` en la raíz del repo (ignorado por git) para
que los entry points lo lean.

In [31]:
if EN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    for var in ("GROQ_API_KEY", "BIOPORTAL_API_KEY"):
        try:
            os.environ[var] = userdata.get(var)
        except Exception:
            pass
else:
    try:
        from dotenv import load_dotenv
        for p in [REPO_DIR / ".env", REPO_DIR / "M3/generacion/.env", REPO_DIR / "M2/harness/.env"]:
            if p.exists():
                load_dotenv(p, override=False)
    except Exception as e:
        print("Aviso cargando .env:", e)

lineas = ["PROJECT_ROOT=" + str(REPO_DIR)]
for var in ("GROQ_API_KEY", "BIOPORTAL_API_KEY"):
    if os.environ.get(var):
        lineas.append(var + "=" + os.environ[var])
(REPO_DIR / ".env").write_text("\n".join(lineas) + "\n", encoding="utf-8")
print("Claves presentes:", [v for v in ("GROQ_API_KEY", "BIOPORTAL_API_KEY") if os.environ.get(v)])
print(".env escrito en", REPO_DIR / ".env")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Claves presentes: ['GROQ_API_KEY', 'BIOPORTAL_API_KEY']
.env escrito en /content/Topicos-IA/.env


## 5 · Configuración del retrieval según el entorno

El YAML del repo (`config_retrieval.yaml`) apunta a Drive para Colab. En **local**
se arma un YAML equivalente (sin tocar el del repo) con el perfil `real` apuntando
a `DATA_ROOT`, y se exporta `M3_RETRIEVAL_CONFIG` para que los módulos lo usen.

In [32]:
import yaml

os.environ["M3_REPO_ROOT"] = str(REPO_DIR)
os.environ["M3_DATA_ROOT"] = str(DATA_ROOT)

# Sin BIOPORTAL_API_KEY, la tool real de Luis cae a un mock diminuto que no conoce
# las siglas del eval set (DM2, HTA, EPOC...). En ese caso se usa el mapa simulado
# del equipo para que la normalizacion funcione en la corrida de demostracion.
mapa_mock = None if os.environ.get("BIOPORTAL_API_KEY") else "M3/retrieval/data/mock/normalizacion_mock.json"


def _cfg_retrieval():
    return yaml.safe_load((REPO_DIR / "M3/retrieval/config_retrieval.yaml").read_text(encoding="utf-8"))


if EN_COLAB:
    if mapa_mock:
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        cfg_r = _cfg_retrieval()
        cfg_r["rutas"]["real"]["mapa_normalizacion"] = mapa_mock
        tmp = DATA_ROOT / "config_retrieval.colab.yaml"
        tmp.write_text(yaml.safe_dump(cfg_r, allow_unicode=True, sort_keys=False), encoding="utf-8")
        os.environ["M3_RETRIEVAL_CONFIG"] = str(tmp)
        print("Sin BIOPORTAL: se usa el mapa de normalizacion simulado.")
    else:
        print("Colab: se usa M3/retrieval/config_retrieval.yaml (rutas de Drive).")
else:
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    cfg_r = _cfg_retrieval()
    base = str(DATA_ROOT)
    cfg_r["rutas"]["real"].update({
        "corpus_json": base + "/data/guias_clinicas/chunks",
        "chroma_origen": base + "/data/chroma_guias",
        "chroma_dir": base + "/chroma_local",
        "manifest": base + "/data/guias_clinicas/corpus_manifest.json",
        "salida": "M3/retrieval/outputs",
    })
    if mapa_mock:
        cfg_r["rutas"]["real"]["mapa_normalizacion"] = mapa_mock
    cfg_r["rutas"]["mock"]["chroma_dir"] = base + "/chroma_mock"
    local_cfg = DATA_ROOT / "config_retrieval.local.yaml"
    local_cfg.write_text(yaml.safe_dump(cfg_r, allow_unicode=True, sort_keys=False), encoding="utf-8")
    os.environ["M3_RETRIEVAL_CONFIG"] = str(local_cfg)
    print("YAML de retrieval local:", local_cfg)



Colab: se usa M3/retrieval/config_retrieval.yaml (rutas de Drive).


## 6 · Verificar recursos

Comprueba las guías (PDF), el resultado de M2 para el cruce y el eval set.

In [33]:
# 1. Crear la carpeta esperada en el repo
!mkdir -p /content/Topicos-IA/M2/ejecucion/outputs_gold

# 2. Copiar el resultado del harness de visión (limpio) a esa ruta
!cp /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/outputs_vision/limpio_vlm/resultado_dimension1.json /content/Topicos-IA/M2/ejecucion/outputs_gold/resultado_dimension1.json

print("¡Listo! El resultado de Visión Limpia quedó enlazado para M3.")

¡Listo! El resultado de Visión Limpia quedó enlazado para M3.


In [34]:
guia_dir = DATA_ROOT / "data/guias_clinicas"
checks = {
    "Guías PDF (data/guias_clinicas)": guia_dir if guia_dir.exists() else (REPO_DIR / "M3/corpus/guias_clinicas"),
    "Resultado M2 (para el cruce)": REPO_DIR / "M2/ejecucion/outputs_gold/resultado_dimension1.json",
    "Eval set del equipo": REPO_DIR / EJEMPLOS,
}
for nombre, p in checks.items():
    print("%s: %s -- %s" % (nombre, "OK" if p.exists() else "FALTA", p))

ejemplos = REPO_DIR / EJEMPLOS
if ejemplos.exists():
    ENTIDADES = EJEMPLOS
elif (REPO_DIR / EJEMPLOS_ALT).exists():
    ENTIDADES = EJEMPLOS_ALT
    print("Aviso: no existe %s; se usa %s" % (EJEMPLOS, EJEMPLOS_ALT))
else:
    raise FileNotFoundError("No hay eval set de entidades.")

Guías PDF (data/guias_clinicas): OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3/data/guias_clinicas
Resultado M2 (para el cruce): OK -- /content/Topicos-IA/M2/ejecucion/outputs_gold/resultado_dimension1.json
Eval set del equipo: OK -- /content/Topicos-IA/M3/ejecucion/eval_set_casos.json


## 7 · Pruebas unitarias

Rápidas y sin descargar modelos. (Algunas pruebas de retrieval dependen del perfil
`corpus` activo en el YAML; pueden fallar si el YAML trae `real`.)

In [35]:
ejecutar([sys.executable, "-m", "unittest", "test_retrieval"], cwd=REPO_DIR / "M3/retrieval", check=False)
ejecutar([sys.executable, "-m", "unittest", "test_generacion"], cwd=REPO_DIR / "M3/generacion", check=False)

$ /usr/bin/python3 -m unittest test_retrieval
Aviso: no hay umbrales calibrados en /tmp/tmp4ork4tw6/umbrales.json; se usan los provisionales del YAML.
Aviso: no hay umbrales calibrados en /tmp/tmpt04h97qb/umbrales.json; se usan los provisionales del YAML.
Índice reconstruido en /tmp/tmp_s3sf0dj/mock/chroma: 4 chunks, 1 guías.
Índice reconstruido en /tmp/tmp_s3sf0dj/real/chroma: 4 chunks, 1 guías.
Aviso: se reconstruye el índice de /tmp/tmpw0fwjlsq/chroma (El índice no coincide con los chunks JSON: el índice tiene 7 chunks y la carpeta de chunks 4; 3 ids del índice no están en los JSON (p. ej. ['epoc_c0', 'sepsis_c0', 'sifilis_c0']).)
Índice reconstruido en /tmp/tmpw0fwjlsq/chroma: 4 chunks, 1 guías.

...................................................
----------------------------------------------------------------------
Ran 51 tests in 0.126s

OK

$ /usr/bin/python3 -m unittest test_generacion
[03:35:05] Piezas REALES: retrieval (M3/retrieval/config_retrieval.yaml) + normalizacion (co

CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'test_generacion'], returncode=0, stdout="[03:35:05] Piezas REALES: retrieval (M3/retrieval/config_retrieval.yaml) + normalizacion (corpus 'mock').\n    [orquestacion] falló el retrieval ('neumonia'): indice caido\n    [orquestacion] falló el retrieval ('ENFERMEDAD CONTROLADA'): indice caido\n", stderr='................\n----------------------------------------------------------------------\nRan 16 tests in 0.005s\n\nOK\n')

## 8 · Corpus (Isa): ingesta de las guías + índice

Ingesta → chunking → embeddings → Chroma + manifest. Es **idempotente**: las
fuentes ya indexadas se saltan por caché de `doc_id`.

In [36]:
if CORRER_CORPUS:
    import shutil
    destino = DATA_ROOT / "data/guias_clinicas"
    destino.mkdir(parents=True, exist_ok=True)
    for pdf in sorted((REPO_DIR / "M3/corpus/guias_clinicas").glob("*.pdf")):
        if not (destino / pdf.name).exists():
            shutil.copy2(pdf, destino / pdf.name)
            print("copiada:", pdf.name)
    ejecutar([sys.executable, "run_corpus.py", "--config", "config.yaml", "--fuentes", "fuentes.yaml"],
             cwd=REPO_DIR / "M3/corpus", extra_env={"PROJECT_ROOT": str(DATA_ROOT)})
else:
    print("CORRER_CORPUS=False: se usa el índice ya construido.")

$ /usr/bin/python3 run_corpus.py --config config.yaml --fuentes fuentes.yaml
CACHE gpc_diabetes_mellitus_tipo_2: chunks ya existen en disco, se omite reprocesamiento
CACHE gpc_falla_cardiaca: chunks ya existen en disco, se omite reprocesamiento
CACHE gpc_depresion_adultos: chunks ya existen en disco, se omite reprocesamiento
{'n_fuentes_ok': 3, 'n_fuentes_cacheadas': 3, 'n_fuentes_nuevas': 0, 'n_fuentes_fallidas': 0, 'n_documentos': 3, 'n_chunks': 64, 'n_secciones_filtradas': 0, 'errores': []}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5954.33it/s]



## 9 · Retrieval (Pau): calibración de umbrales y delta

Experimento controlado (ingenuo vs. híbrido vs. reranker vs. sistema completo).
Escribe `umbrales.json` y `deltas_s08.csv` en `M3/retrieval/outputs/`.

In [37]:
if CORRER_EXPERIMENTO:
    r = ejecutar([sys.executable, "experimento_s08.py", "--corpus", "real"],
                 cwd=REPO_DIR / "M3/retrieval", check=False)
    if r.returncode:
        print("\nAVISO: el experimento de calibracion fallo "
              "(las consultas del corpus real pueden estar sin etiquetar). "
              "Se continua con los umbrales provisionales del YAML.")
else:
    print("CORRER_EXPERIMENTO=False: se usan los umbrales provisionales del YAML.")


$ /usr/bin/python3 experimento_s08.py --corpus real
Denso (RAG ingenuo) | 22 | 0.955 | 1.0 | 0.97 | 1.0 | 0.936 | 0.045 | 0.409 | 0.161 | 0.118 | 0.0 | 0.0 | - | 13.8 | ingenuo | 0.0 | 0.0
Híbrido BM25 + denso | 22 | 0.955 | 1.0 | 0.97 | 1.0 | 0.945 | 0.136 | 0.409 | 0.239 | 0.127 | 0.0 | 0.0 | - | 12.7 | hibrido | 0.0 | 0.078
Híbrido + reranker | 22 | 0.909 | 0.955 | 0.932 | 0.955 | 0.936 | 0.273 | 0.864 | 0.502 | 0.255 | 0.0 | 0.0 | - | 92.1 | reranker | -0.038 | 0.341
Híbrido + reranker con pregunta de tratamiento | 22 | 0.909 | 0.909 | 0.909 | 0.909 | 0.909 | 0.636 | 0.909 | 0.761 | 0.473 | 0.0 | 0.0 | - | 91.5 | reranker_pregunta | -0.061 | 0.6
Consultas de tratamiento, sin reranker | 22 | 0.909 | 1.0 | 0.955 | 1.0 | 0.9 | 0.182 | 0.682 | 0.34 | 0.227 | 0.0 | 0.0 | - | 38.7 | intencion_sin_reranker | -0.015 | 0.179
Consultas de tratamiento + reranker | 22 | 0.909 | 0.909 | 0.909 | 0.909 | 0.909 | 0.682 | 0.909 | 0.784 | 0.491 | 0.0 | 0.0 | - | 117.0 | intencion | -0.061 | 0.623
Co

## 10 · Generación (Agustín): respuesta final del RAG

Encadena orquestación (cuándo se invoca la tool y qué hace si falla) + generación.
Consume el retrieval de Pau y la tool de Luis desde sus YAML y escribe
`M3/outputs/resultado_generacion.json`, que consume RAGAS.

In [38]:
ejecutar([sys.executable, "run_generacion.py", "--config", "config.yaml", "--entidades", ENTIDADES],
         cwd=REPO_DIR / "M3/generacion", extra_env={"PROJECT_ROOT": str(REPO_DIR)})

$ /usr/bin/python3 run_generacion.py --config config.yaml --entidades M3/ejecucion/eval_set_casos.json
[03:36:19] PROJECT_ROOT: /content/Topicos-IA
[03:36:19] Entidades: 15 (desde M3/ejecucion/eval_set_casos.json)
[03:36:39] Piezas REALES: retrieval (M3/retrieval/config_retrieval.yaml) + normalizacion (corpus 'real').
[03:36:39] Generando sobre 15 entidades (umbral=0.174136, umbral_evidencia=0.174136, k=5, modelo=qwen/qwen3.8-27b)...
[03:36:43]   [1/15] 'diabetes mellitus tipo 2' (estandar) -> query='diabetes mellitus tipo 2' | tool=False | fragmentos=5 | fallback=False | fuentes=5
[03:36:56]   [2/15] 'DM2' (abreviatura) -> query='diabetes mellitus tipo 2' | tool=True | fragmentos=5 | fallback=True | fuentes=0
[03:37:00]   [3/15] 'diabetes mellitus tipo 2' (urgencia) -> query='diabetes mellitus tipo 2' | tool=False | fragmentos=5 | fallback=False | fuentes=5
[03:37:04]   [4/15] 'DM2' (multimorbilidad) -> query='diabetes mellitus tipo 2' | tool=True | fragmentos=5 | fallback=True | fuen

CompletedProcess(args=['/usr/bin/python3', 'run_generacion.py', '--config', 'config.yaml', '--entidades', 'M3/ejecucion/eval_set_casos.json'], returncode=0, stdout="[03:36:19] PROJECT_ROOT: /content/Topicos-IA\n[03:36:19] Entidades: 15 (desde M3/ejecucion/eval_set_casos.json)\n[03:36:39] Piezas REALES: retrieval (M3/retrieval/config_retrieval.yaml) + normalizacion (corpus 'real').\n[03:36:39] Generando sobre 15 entidades (umbral=0.174136, umbral_evidencia=0.174136, k=5, modelo=qwen/qwen3.8-27b)...\n[03:36:43]   [1/15] 'diabetes mellitus tipo 2' (estandar) -> query='diabetes mellitus tipo 2' | tool=False | fragmentos=5 | fallback=False | fuentes=5\n[03:36:56]   [2/15] 'DM2' (abreviatura) -> query='diabetes mellitus tipo 2' | tool=True | fragmentos=5 | fallback=True | fuentes=0\n[03:37:00]   [3/15] 'diabetes mellitus tipo 2' (urgencia) -> query='diabetes mellitus tipo 2' | tool=False | fragmentos=5 | fallback=False | fuentes=5\n[03:37:04]   [4/15] 'DM2' (multimorbilidad) -> query='diabet

## 11 · RAGAS + cruce con el harness de M2 (Luis)

Faithfulness, context precision, context recall y answer relevancy, cruzados con
el F1 de extracción de M2 para diagnosticar dónde falla el sistema.

In [39]:
ejecutar([sys.executable, "run_ragas.py", "--config", "config.yaml", "--modo", MODO_RAGAS],
         cwd=REPO_DIR / "M3/ragas", extra_env={"PROJECT_ROOT": str(REPO_DIR)})

$ /usr/bin/python3 run_ragas.py --config config.yaml --modo real
[ragas] Cargados 15 casos reales desde /content/Topicos-IA/M3/outputs/resultado_generacion.json
[ragas] Modo: 'real' | Fuente de datos: /content/Topicos-IA/M3/outputs/resultado_generacion.json | Casos: 15

--- Evaluación: ragas (parcial) ---
  faithfulness: N/D (0/15 valores calculados)
  context_precision: 0.0000 (1/15 valores calculados)
  context_recall: N/D (0/15 valores calculados)
  answer_relevancy: 0.8052 (2/15 valores calculados)

RESULTADOS M2 Y M3: ALCANCE SEPARADO
### M2: extracción por documento
| doc_id | F1 extracción |
|---|---|
| caso_01 | 1.000 |
| caso_02 | 0.857 |
| caso_03 | 0.600 |
| caso_04 | 0.857 |
| caso_05 | 0.800 |
| caso_06 | 0.545 |
| caso_07 | 0.400 |
| caso_08 | 0.667 |
| caso_09 | 0.000 |
| caso_10 | 0.500 |
| caso_11 | 0.250 |
| caso_12 | 0.750 |
| caso_13 | 0.400 |
| caso_14 | 0.667 |
| caso_15 | 0.750 |

### M3: métricas agregadas de la corrida de evaluación
| Métrica | Valor |
|---|---

CompletedProcess(args=['/usr/bin/python3', 'run_ragas.py', '--config', 'config.yaml', '--modo', 'real'], returncode=0, stdout='[ragas] Cargados 15 casos reales desde /content/Topicos-IA/M3/outputs/resultado_generacion.json\n[ragas] Modo: \'real\' | Fuente de datos: /content/Topicos-IA/M3/outputs/resultado_generacion.json | Casos: 15\n\n--- Evaluación: ragas (parcial) ---\n  faithfulness: N/D (0/15 valores calculados)\n  context_precision: 0.0000 (1/15 valores calculados)\n  context_recall: N/D (0/15 valores calculados)\n  answer_relevancy: 0.8052 (2/15 valores calculados)\n\n============================================================\nRESULTADOS M2 Y M3: ALCANCE SEPARADO\n============================================================\n### M2: extracción por documento\n| doc_id | F1 extracción |\n|---|---|\n| caso_01 | 1.000 |\n| caso_02 | 0.857 |\n| caso_03 | 0.600 |\n| caso_04 | 0.857 |\n| caso_05 | 0.800 |\n| caso_06 | 0.545 |\n| caso_07 | 0.400 |\n| caso_08 | 0.667 |\n| caso_09 | 0.0

## 12 · Persistir los artefactos

Copia los resultados a `DATA_ROOT/resultados_pipeline` y arma un zip. En Colab
también ofrece la descarga.

In [40]:
import shutil

persist = DATA_ROOT / "resultados_pipeline"
persist.mkdir(parents=True, exist_ok=True)
for origen, nombre in [(REPO_DIR / "M3/outputs", "generacion"),
                       (REPO_DIR / "M3/retrieval/outputs", "retrieval")]:
    if origen.exists():
        shutil.copytree(origen, persist / nombre, dirs_exist_ok=True)
        print("copiado:", persist / nombre)

zip_path = shutil.make_archive(str(DATA_ROOT / "resultados_M3"), "zip", str(persist))
print("zip:", zip_path)

if EN_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("Descarga manual en:", zip_path, "|", e)

copiado: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3/resultados_pipeline/generacion
copiado: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3/resultados_pipeline/retrieval
zip: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3/resultados_M3.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>